<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-2-grafos/12%20-%20Matrizes%20de%20Incidencia%20Adjacencia%20e%20Custos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 12 - Notebook: Matriz de Incidência e Balanço Hidráulico Matricial na Linha de Envase

Neste notebook implementamos a **Matriz de Incidência Vértice-Aresta** $B \in \{-1, 0, 1\}^{n \times m}$ e resolvemos o balanço de vazão volumétrica matricial $B \cdot \vec{Q} = \vec{S}$ para a rede da **Linha de Envasamento de Bebidas (SCADA-Core - Grupo 3)**.


In [1]:
from typing import List, Dict, Tuple, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

def formatar_matriz(matriz: List[List[Any]], rotulos_linhas: List[str], rotulos_cols: List[str]) -> str:
    """Formata matriz 2D em tabela ASCII pura."""
    larguras = [max(len(str(r)), 6) for r in rotulos_cols]
    larg_linha = max(len(str(r)) for r in rotulos_linhas)
    header = f"{' ' * larg_linha} | " + " | ".join(f"{c:>{larguras[j]}}" for j, c in enumerate(rotulos_cols))
    divisor = f"{'-' * larg_linha}-+-" + "-+-".join("-" * larguras[j] for j in range(len(rotulos_cols)))
    linhas = [header, divisor]
    for i, r_nome in enumerate(rotulos_linhas):
        vals = []
        for j in range(len(rotulos_cols)):
            v = matriz[i][j]
            v_str = "∞" if v == float('inf') else str(v)
            vals.append(f"{v_str:>{larguras[j]}}")
        linhas.append(f"{r_nome:<{larg_linha}} | " + " | ".join(vals))
    return "\n".join(linhas)

class GrafoTubulacao:
    def __init__(self, vertices: List[str]):
        self.vertices = vertices
        self.v_to_idx = {v: i for i, v in enumerate(vertices)}
        self.idx_to_v = {i: v for i, v in enumerate(vertices)}
        self.n = len(vertices)
        self.adj_binaria = [[0] * self.n for _ in range(self.n)]
        self.adj_pesos = [[float('inf')] * self.n for _ in range(self.n)]
        for i in range(self.n):
            self.adj_pesos[i][i] = 0.0
        self.arestas_detalhes: List[Dict[str, Any]] = []

    def adicionar_tubulacao(self, origem: str, destino: str, comprimento_m: float, tag_valvula: str, diametro_pol: float = 3.0):
        u = self.v_to_idx[origem]
        v = self.v_to_idx[destino]
        self.adj_binaria[u][v] = 1
        self.adj_pesos[u][v] = comprimento_m
        self.arestas_detalhes.append({
            "Origem": origem,
            "Destino": destino,
            "Comprimento (m)": comprimento_m,
            "Válvula ISA": tag_valvula,
            "Diâmetro (pol)": diametro_pol
        })

def criar_rede_envase() -> GrafoTubulacao:
    nos_envase = [
        "TS1_Suprimento",
        "VS1_Succao",
        "BC1_Bomba",
        "AS1_Acumulador",
        "VS2_Envase",
        "SQ2_Medicao",
        "EST_Envase",
        "VALV_Alivio"
    ]
    g = GrafoTubulacao(nos_envase)
    g.adicionar_tubulacao("TS1_Suprimento", "VS1_Succao", 5.0, "VS1", 3.0)
    g.adicionar_tubulacao("VS1_Succao", "BC1_Bomba", 3.0, "SP1_SQ1", 3.0)
    g.adicionar_tubulacao("BC1_Bomba", "AS1_Acumulador", 8.0, "CHECK_V", 2.5)
    g.adicionar_tubulacao("AS1_Acumulador", "VS2_Envase", 10.0, "VS2", 2.0)
    g.adicionar_tubulacao("AS1_Acumulador", "VALV_Alivio", 6.0, "SP2_VS3", 2.0)
    g.adicionar_tubulacao("VALV_Alivio", "TS1_Suprimento", 12.0, "RET_V", 2.0)
    g.adicionar_tubulacao("VS2_Envase", "SQ2_Medicao", 2.0, "SQ2_IN", 1.5)
    g.adicionar_tubulacao("SQ2_Medicao", "EST_Envase", 1.5, "BICO_FILL", 1.5)
    g.adicionar_tubulacao("AS1_Acumulador", "SQ2_Medicao", 11.0, "XV_BYPASS", 2.0)
    return g

rede = criar_rede_envase()

class CalculadorIncidencia:
    @staticmethod
    def construir_matriz_incidencia(vertices: List[str], arestas: List[Dict[str, Any]]) -> Tuple[List[List[int]], List[str]]:
        n = len(vertices)
        m = len(arestas)
        v_idx = {v: i for i, v in enumerate(vertices)}
        B = [[0] * m for _ in range(n)]
        nomes_e = []
        for j, a in enumerate(arestas):
            u = v_idx[a["Origem"]]
            v = v_idx[a["Destino"]]
            B[u][j] = -1
            B[v][j] = 1
            nomes_e.append(f"e{j+1}:{a['Origem']}->{a['Destino']}")
        return B, nomes_e

# Construção da Matriz B
B_mat, nomes_arestas = CalculadorIncidencia.construir_matriz_incidencia(rede.vertices, rede.arestas_detalhes)
print("=== MATRIZ DE INCIDÊNCIA VÉRTICE-ARESTA (B) ===")
print(formatar_matriz(B_mat, rede.vertices, [f"e{j+1}" for j in range(len(rede.arestas_detalhes))]))

# Verificação da propriedade formal da soma nula de colunas
somas_col = [sum(B_mat[i][j] for i in range(len(rede.vertices))) for j in range(len(rede.arestas_detalhes))]
assert all(s == 0 for s in somas_col), "Erro: A soma das colunas da matriz de incidência deve ser nula!"
print("\n✓ Verificação formal concluída: Todas as colunas possuem soma exatamente NULA (Conservação de Vazão).")

# Simulação de Balanço Hidráulico de Envase (Vetor Q em L/min)
# e1..e9: [TS1->VS1, VS1->BC1, BC1->AS1, AS1->VS2, AS1->VALV_Alivio, VALV_Alivio->TS1, VS2->SQ2, SQ2->EST_Envase, AS1->SQ2_Bypass]
Q_vec = [15.0, 15.0, 15.0, 12.0, 3.0, 3.0, 12.0, 12.0, 0.0]

S_res = []
for i, v_nome in enumerate(rede.vertices):
    balanco = sum(B_mat[i][j] * Q_vec[j] for j in range(len(Q_vec)))
    S_res.append({"Componente / Nó": v_nome, "Balanço Líquido de Vazão (L/min)": f"{balanco:.1f}"})

print("\n--- Balanço Hidráulico em Regime Permanente de Envase (S = B * Q) ---")
print(formatar_tabela(S_res))


=== MATRIZ DE INCIDÊNCIA VÉRTICE-ARESTA (B) ===
               |     e1 |     e2 |     e3 |     e4 |     e5 |     e6 |     e7 |     e8 |     e9
---------------+--------+--------+--------+--------+--------+--------+--------+--------+-------
TS1_Suprimento |     -1 |      0 |      0 |      0 |      0 |      1 |      0 |      0 |      0
VS1_Succao     |      1 |     -1 |      0 |      0 |      0 |      0 |      0 |      0 |      0
BC1_Bomba      |      0 |      1 |     -1 |      0 |      0 |      0 |      0 |      0 |      0
AS1_Acumulador |      0 |      0 |      1 |     -1 |     -1 |      0 |      0 |      0 |     -1
VS2_Envase     |      0 |      0 |      0 |      1 |      0 |      0 |     -1 |      0 |      0
SQ2_Medicao    |      0 |      0 |      0 |      0 |      0 |      0 |      1 |     -1 |      1
EST_Envase     |      0 |      0 |      0 |      0 |      0 |      0 |      0 |      1 |      0
VALV_Alivio    |      0 |      0 |      0 |      0 |      1 |     -1 |      0 |      0 |